# Limpieza de archivos Probabilidad y Estadística

Cargar archivos de datos

In [1]:
from pathlib import Path
import pandas as pd
from utils_limpieza import *

archivos = sorted(Path("../data/probabilidadestadistica").glob("calificaciones*.xlsx"))
dfs = {a.name: pd.read_excel(a, header=6) for a in archivos}

Limpieza inicial de las calificaciones: nombres de columnas y filas nulas

In [2]:
for nombre, df in dfs.items():
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    filas_nulas = df.isna().all(axis=1)
    sin_nombre = df[["apellido", "nombre"]].isna().any(axis=1) & ~filas_nulas
    sin_usuario = df["usuario"].isna() & ~filas_nulas
    print(nombre)
    print(f"  Filas totalmente nulas: {filas_nulas.sum()}")
    print(f"  Filas sin nombre/apellido: {sin_nombre.sum()}")
    print(f"  Filas sin usuario: {sin_usuario.sum()}")

    df = df.dropna(how="all").dropna(axis=1, how="all").dropna(subset=["usuario", "apellido", "nombre"])
    dfs[nombre] = filtrar_y_estandarizar_usuarios(df, nombre)

calificaciones_alumnos_20251024_115041.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115041.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_115045.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115045.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_115049.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115049.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_115055.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115055.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_115059.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115059.xlsx: staff/docentes excluidos = 0


Separación de la nota

In [3]:
NOTA_MAX = 100

for nombre, df in dfs.items():
    print(f"{nombre}: notas nulas = {df['nota'].isna().sum()}")
    df["nota"] = df["nota"].fillna(f"0/{NOTA_MAX}")
    df[["nota_obtenida", "nota_max"]] = df["nota"].str.split("/", expand=True).astype(float)
    dfs[nombre] = validar_notas(df, nombre, {NOTA_MAX})

calificaciones_alumnos_20251024_115041.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115041.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115045.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115045.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115049.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115049.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115055.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115055.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115059.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115059.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115102.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115102.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_115105.xlsx: n

Eliminación de duplicados, unificación de calificaciones y exportación a Parquet

In [4]:
for nombre, df in dfs.items():
    duplicadas = df.duplicated().sum()
    usuarios_repetidos = df["usuario"].duplicated().sum()
    print(f"{nombre}: filas duplicadas = {duplicadas}, usuarios repetidos = {usuarios_repetidos}")

    dfs[nombre] = (df.drop_duplicates()
                   .sort_values("nota_obtenida", ascending=False)
                   .drop_duplicates(subset="usuario")
                   .sort_index())

df_calificaciones = pd.concat(dfs, names=["archivo"]).reset_index(level="archivo").reset_index(drop=True)
df_calificaciones = df_calificaciones.drop(columns=["nota"], errors="ignore")

df_calificaciones.to_parquet("../data/probabilidadestadistica/calificaciones_probabilidadestadistica.parquet", index=False)
df_calificaciones

calificaciones_alumnos_20251024_115041.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115045.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115049.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115055.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115059.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115102.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115105.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115127.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115130.xlsx: filas duplicadas = 0, usuarios repetidos = 0


,archivo,usuario,apellido,nombre,observaciones,nota_obtenida,nota_max
0,calificaciones_alumnos_20251024_115041.xlsx,ALU_1001,Gómez,Manuel,aprobada,65.00,100.0
1,calificaciones_alumnos_20251024_115041.xlsx,ALU_1002,Silva,Diego,aprobada,70.00,100.0
2,calificaciones_alumnos_20251024_115041.xlsx,ALU_1004,Flores,Gabriel,aprobada,95.00,100.0
3,calificaciones_alumnos_20251024_115041.xlsx,ALU_1005,Vega,Agustín,aprobada,65.00,100.0
4,calificaciones_alumnos_20251024_115041.xlsx,ALU_1006,Ramos,Patricio,aprobada,90.00,100.0
...,...,...,...,...,...,...,...
890,calificaciones_alumnos_20251024_115130.xlsx,ALU_1179,Alvarez,Victoria,aprobada,100.00,100.0
891,calificaciones_alumnos_20251024_115130.xlsx,ALU_1224,López,Iara,aprobada,83.33,100.0
892,calificaciones_alumnos_20251024_115130.xlsx,ALU_1238,Vargas,Damián,aprobada,91.67,100.0
893,calificaciones_alumnos_20251024_115130.xlsx,ALU_1277,Silva,Gisela,aprobada,100.00,100.0


Carga y limpieza del reporte de participación

In [5]:
archivo = next(Path("../data/probabilidadestadistica").glob("reporte_participacion*.xlsx"))
df_participacion = pd.read_excel(archivo, header=[5, 6])
# Encabezado de dos niveles (ej. "Encuestas" / "Cantidad"): se unen en un solo nombre
df_participacion.columns = pd.Index(
    [" ".join(n for n in col if not n.startswith("Unnamed")) for col in df_participacion.columns]
)

df_participacion = limpiar_reporte(df_participacion, "participacion")
df_participacion = procesar_participacion(df_participacion)

df_participacion.to_parquet("../data/probabilidadestadistica/participacion_probabilidadestadistica.parquet", index=False)
df_participacion

participacion: staff/docentes excluidos = 9
participacion:
  Fechas inconsistentes (última < primera, o primera < alta): 0
  Filas con valores numéricos negativos: 0
  Avance fuera de 0-100: 0
  Aulas: ['TUCD-MD-Probabilidad y Estadística 2025']
  Perfiles: {'Alumnos': 319}
  Filas duplicadas: 0, usuarios repetidos: 0
  Nulos por columna: {'primera_interaccion': 46, 'ultima_interaccion': 46}


,usuario,apellido,nombre,aula,perfil,fecha_alta,avance,interacciones,primera_interaccion,ultima_interaccion,...,foros_temas_de_debate_con_intervencion,foros_cantidad_de_intervenciones_del_usuario,anuncios_cantidad,anuncios_cargados,noticias_cantidad,noticias_cargados,tiempo_estimado_de_uso_min,fechas_inconsistentes,valores_negativos,avance_fuera_de_rango
0,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,2025-08-05,80,1071,2025-08-06 18:04:07,2025-10-23 21:49:01,...,0,0,0,0,11,0,1079.0,False,False,False
1,ALU_1002,Silva,Diego,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,2025-08-05,85,1067,2025-08-05 13:21:48,2025-10-22 21:47:58,...,0,0,0,0,11,0,1157.0,False,False,False
2,ALU_1003,Castro,Federico,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,2025-08-05,1,11,2025-08-05 17:40:21,2025-08-06 20:02:40,...,0,0,0,0,11,0,5.0,False,False,False
3,ALU_1004,Flores,Gabriel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,2025-08-05,94,1080,2025-08-05 12:01:33,2025-10-23 09:38:54,...,0,0,0,0,11,0,1082.0,False,False,False
4,ALU_1005,Vega,Agustín,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,2025-08-05,92,640,2025-08-05 11:55:16,2025-10-22 15:30:28,...,0,0,0,0,11,0,663.0,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,ALU_1315,Flores,Felipe,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,2025-08-05,84,394,2025-08-05 22:46:40,2025-10-21 20:18:18,...,0,0,0,0,11,0,282.0,False,False,False
315,ALU_1316,Sosa,Emilia,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,2025-08-05,20,220,2025-08-06 19:24:39,2025-10-19 22:33:12,...,0,0,0,0,11,0,188.0,False,False,False
316,ALU_1317,Morales,Nahuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,2025-08-05,0,0,NaT,NaT,...,0,0,0,0,11,0,0.0,False,False,False
317,ALU_1318,Villalba,Candelaria,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,2025-08-05,24,53,2025-08-06 14:40:44,2025-10-14 12:26:08,...,0,0,0,0,11,0,36.0,False,False,False


Carga y limpieza del reporte de trazabilidad

In [6]:
archivo = next(Path("../data/probabilidadestadistica").glob("reporte_trazabilidad*.xlsx"))
df_trazabilidad = procesar_trazabilidad(limpiar_reporte(pd.read_excel(archivo, header=5), "trazabilidad"))

df_trazabilidad.to_parquet("../data/probabilidadestadistica/trazabilidad_probabilidadestadistica.parquet", index=False)
df_trazabilidad

trazabilidad: staff/docentes excluidos = 4558
trazabilidad:
  Eventos sin fecha válida (descartados): 0
  Rango de fechas: 2025-08-05 10:21:08 -> 2025-10-24 11:38:20
  Aulas: ['TUCD-MD-Probabilidad y Estadística 2025']
  Perfiles: {'Alumnos': 63578}
  Filas duplicadas (descartadas): 285
  Nulos por columna: {'accion': 10}


,fecha,ip_usuario,usuario,apellido,nombre,aula,perfil,seccion,accion,descripcion
0,2025-08-06 18:04:07,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Inicio,Acceder,Acceso a la sección 'Inicio'
1,2025-08-06 18:04:30,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Noticias,Ver noticia,Visualización de la noticia '&#128221; Encuest...
2,2025-08-06 18:04:40,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Noticias,Acceder,Acceso a la sección 'Noticias'
3,2025-08-06 18:04:46,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Noticias,Ver noticia,Visualización de la noticia '&#128226; ¡Bienve...
4,2025-08-06 18:05:04,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Calendario,Acceder,Acceso a la sección 'Calendario'
...,...,...,...,...,...,...,...,...,...,...
63288,2025-10-22 20:00:14,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Programa > Material de estudio,Ver archivo,Acceso al material de estudio 'GUIA_Probabilid...
63289,2025-10-22 20:00:22,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Programa > Material de estudio,Descargar,Descarga del material de estudio 'GUIA_Probabi...
63290,2025-10-22 20:00:38,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Programa,Acceder,Acceso a la sección 'Programa'
63291,2025-10-22 20:00:45,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Programa > Textos,Ver texto,Acceso al texto 'Link a Clases en Teams : Sema...


Cruce con Matriculados

In [7]:
df_matriculados = pd.read_parquet("../data/matriculados/matriculados.parquet")
df_cruce = cruzar_con_matriculados(df_matriculados, "probabilidad", {
    "trazabilidad": df_trazabilidad,
    "participacion": df_participacion,
    "calificaciones": df_calificaciones,
})
df_cruce.to_parquet("../data/probabilidadestadistica/cruce_matriculados_probabilidadestadistica.parquet", index=False)
df_cruce[df_cruce["activo_no_inscripto"]]

Matriculados (probabilidad): 167
  trazabilidad: 273 usuarios, 109 no matriculados
  participacion: 319 usuarios, 152 no matriculados
  calificaciones: 194 usuarios, 52 no matriculados
Activos no inscriptos: 109
Activos con estado Baja: 3
Inscriptos sin actividad: 3


,usuario,matriculado,en_trazabilidad,en_participacion,en_calificaciones,estado_insc,instancia,activo_no_inscripto,activo_con_baja,inscripto_sin_actividad
2,ALU_1003,False,True,True,False,<NA>,<NA>,True,False,False
4,ALU_1005,False,True,True,True,<NA>,<NA>,True,False,False
6,ALU_1007,False,True,True,False,<NA>,<NA>,True,False,False
8,ALU_1009,False,True,True,True,<NA>,<NA>,True,False,False
9,ALU_1010,False,True,True,False,<NA>,<NA>,True,False,False
...,...,...,...,...,...,...,...,...,...,...
310,ALU_1311,False,True,True,False,<NA>,<NA>,True,False,False
312,ALU_1313,False,True,True,False,<NA>,<NA>,True,False,False
314,ALU_1315,False,True,True,True,<NA>,<NA>,True,False,False
317,ALU_1318,False,True,True,False,<NA>,<NA>,True,False,False
